# FairML Toolkit — Part 5: Pipeline Orchestrator Demo

This notebook walks through one complete execution of the **Fairness Pipeline Development Toolkit**.

The pipeline is fully defined by `config.yml` and executed by `run_pipeline.py`.  
No code changes are needed between runs — only the config changes.

---

## What we will do

| Step | Module | What happens |
|---|---|---|
| 1 | MeasurementModule | Audit raw label disparity → **baseline report** |
| 2a | PipelineModule | Apply `DisparateImpactRemover` to repair feature distributions |
| 2b | TrainingModule | Train a `LogisticRegression` under a `DemographicParity` constraint |
| 3 | MeasurementModule | Re-evaluate on test predictions → **report card** |
| — | MLflow | Log all metrics and artifacts for reproducibility |

## 0. Environment check

In [ ]:
import importlib, sys

required = ["yaml", "mlflow", "fairlearn", "sklearn", "pandas", "numpy"]
missing  = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Missing packages: {missing}")
    print("Run:  pip install -r requirements.txt")
else:
    print("✓ All required packages are installed.")
    print(f"  Python {sys.version.split()[0]}")

---

## 1. Inspect `config.yml`

The config is the **single source of truth** for the entire pipeline.  
Every transformer, constraint, threshold, and MLflow setting lives here.

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = "config.yml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print(Path(CONFIG_PATH).read_text())

### Key settings at a glance

In [ ]:
import pandas as pd

summary = {
    "Dataset":           cfg["data"]["path"],
    "Target column":     cfg["data"]["target_col"],
    "Sensitive column":  cfg["data"]["sensitive_col"],
    "Transformer":       cfg["transformer"]["class"],
    "Repair level":      cfg["transformer"]["params"].get("repair_level"),
    "Training method":   cfg["training"]["method"],
    "Constraint":        cfg["training"]["constraint"],
    "Primary metric":    cfg["validation"]["primary_metric"],
    "Pass threshold":    cfg["validation"]["threshold"],
    "MLflow experiment": cfg["mlflow"]["experiment_name"],
}

pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])

---

## 2. Load and preview the dataset

In [ ]:
from run_pipeline import load_config, load_data

cfg = load_config(CONFIG_PATH)
df  = load_data(cfg)

print(f"\nShape : {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
sensitive_col = cfg["data"]["sensitive_col"]
target_col    = cfg["data"]["target_col"]

print("Group distribution (sensitive attribute):")
print(df[sensitive_col].value_counts())
print()
print("Positive-label rate by group:")
print(
    df.groupby(sensitive_col)[target_col]
    .mean()
    .rename("approval_rate")
    .round(4)
)

---

## 3. Step 1 — Baseline fairness audit

`FairnessAnalyzer` measures label-level disparity in the **raw data** before any intervention.  
This is our starting point — the bias we need to reduce.

In [ ]:
from run_pipeline import step1_baseline

baseline_results = step1_baseline(df, cfg)

In [ ]:
primary = cfg["validation"]["primary_metric"]
b       = baseline_results[primary]

print("Baseline FairnessResult:")
print(b)

**Interpreting the baseline:**

- **Value** — the raw demographic parity gap between groups. A value of 0 means perfect parity; larger absolute values indicate greater disparity.
- **95% CI** — bootstrapped confidence interval. If it excludes zero, the disparity is statistically reliable, not random noise.
- **Effect size (Risk Ratio)** — ratio of the minority group's approval rate to the majority group's. Values below 0.8 are typically flagged as discriminatory under the 80% rule.

---

## 4. Step 2 — Data transformation and fair training

Two sub-steps run sequentially:

1. **`DisparateImpactRemover`** (PipelineModule) repairs numeric feature distributions across groups, reducing proxy discrimination in the input data.
2. **`ReductionsWrapper`** (TrainingModule) trains a `LogisticRegression` using fairlearn's `ExponentiatedGradient` algorithm, enforcing a `DemographicParity` constraint throughout optimisation.

In [ ]:
from run_pipeline import step2_transform_and_train

model, X_test_tr, y_test, s_test = step2_transform_and_train(df, cfg)

In [ ]:
import numpy as np

print("Ensemble summary:")
print(f"  Number of predictors : {len(model.predictors_)}")
print(f"  Best predictor weight: {model.best_weight_:.4f}")
print(f"  Top-3 weights        : {sorted(model.weights_, reverse=True)[:3]}")

---

## 5. Step 3 — Final validation and report card

`FairnessAnalyzer` re-evaluates on the **test-set predictions** of the fair model.  
The report card compares the final metric against the baseline and the configured threshold.

In [ ]:
from run_pipeline import step3_validate

final_results, accuracy = step3_validate(
    model, X_test_tr, y_test, s_test, cfg, baseline_results
)

In [ ]:
# Side-by-side summary table
f = final_results[primary]
b = baseline_results[primary]

comparison = pd.DataFrame({
    "Stage":      ["Baseline (raw labels)", "Final (model predictions)"],
    "Value":      [round(b.value, 4),  round(f.value, 4)],
    "CI lower":   [round(b.confidence_interval[0], 4), round(f.confidence_interval[0], 4)],
    "CI upper":   [round(b.confidence_interval[1], 4), round(f.confidence_interval[1], 4)],
    "Effect size":[round(b.effect_size, 4) if b.effect_size else None,
                   round(f.effect_size, 4) if f.effect_size else None],
})

threshold = cfg["validation"]["threshold"]
delta     = f.value - b.value
passed    = abs(f.value) <= threshold

print(f"Primary metric   : {primary}")
print(f"Change           : {delta:+.4f}")
print(f"Threshold        : ≤ {threshold}")
print(f"Validation gate  : {'✅ PASS' if passed else '❌ FAIL'}")
print(f"Test accuracy    : {accuracy:.4f}")
print()
comparison

---

## 6. MLflow — log everything

A single call logs:
- **Metrics** — accuracy, baseline and final fairness metric with CI bounds
- **Params** — transformer, constraint, threshold
- **Artifacts** — the trained model and the exact `config.yml` used

In [ ]:
from run_pipeline import log_to_mlflow

log_to_mlflow(model, baseline_results, final_results, accuracy, cfg, CONFIG_PATH)

In [ ]:
# Preview what was logged
import mlflow

client = mlflow.MlflowClient()
exp    = client.get_experiment_by_name(cfg["mlflow"]["experiment_name"])
runs   = client.search_runs(experiment_ids=[exp.experiment_id],
                             order_by=["start_time DESC"],
                             max_results=1)

if runs:
    run = runs[0]
    print(f"Run ID   : {run.info.run_id}")
    print(f"Run name : {run.data.tags.get('mlflow.runName')}")
    print(f"Status   : {run.info.status}")
    print("\nLogged metrics:")
    for k, v in sorted(run.data.metrics.items()):
        print(f"  {k:<45} {v:.6f}")
    print("\nLogged params:")
    for k, v in sorted(run.data.params.items()):
        print(f"  {k:<30} {v}")

Open the MLflow UI to explore the run visually:

```bash
mlflow ui
# → http://localhost:5000
```

Navigate to the experiment, open the run, and check the **Artifacts** tab to see the logged model and config file.

---

## 7. Run the full pipeline in one call

All of the above is orchestrated by a single entry point.  
In production, a team member runs this — no notebook needed.

In [ ]:
from run_pipeline import main

main(config_path=CONFIG_PATH)

Or from the terminal:

```bash
python run_pipeline.py --config config.yml
```

---

## 8. Key takeaways

| | |
|---|---|
| **Declarative config** | The entire workflow is reproducible from `config.yml` alone — no hardcoded values anywhere in the orchestrator |
| **Two-layer mitigation** | Pre-processing (`DisparateImpactRemover`) and in-processing (`ReductionsWrapper`) work together; neither alone is sufficient |
| **Statistical rigour** | Every fairness metric comes with a bootstrapped 95% CI, so the report card distinguishes real improvement from sampling noise |
| **MLflow traceability** | Every run is fully logged — metrics, params, model artifact, and the exact config used — making results comparable across the organisation |
| **Extendable** | Adding a new transformer or constraint is one line in the registry dict and one line in `config.yml` |